# AI Builders Club - NetworkEvolution | Session 1
## Script 02 : Get the LIVE running config of both switches (SSH)

**Goal:** log in to **sw1** and **sw2** with SSH from Python, run `show running-config` and print it. **No AI yet.**
This is the **data** we will give to the AI in script 03.

| Section | What you build |
|---------|----------------|
| 1 | Imports + load `.env` |
| 2 | One dictionary per switch (values from `.env`) |
| 3 | Quick test: SSH to sw1 and print the config |
| 4 | A reusable function `get_running_config(device)` |
| 5 | Loop over both switches with friendly errors |

### How to use this notebook
Every **code cell** below is one section of the script `02_get_running_config.py`.

1. Create an empty file `02_get_running_config.py` in your session folder.
2. Copy the code cells **one by one, in order**, and paste each one at the **bottom** of that file.
3. After each paste, run it: `python 02_get_running_config.py` - and check the result described above the cell.

> Markdown cells (like this one) are the explanations - do **not** paste them into the `.py` file.

### Big picture: what Netmiko does for you

| What **you** do by hand in a terminal | What **Netmiko** does in Python |
|---------------------------------------|---------------------------------|
| `ssh admin@172.20.20.11` | `ConnectHandler(**device)` |
| type the password | uses the password from the dict |
| wait for the prompt `sw1>` / `sw1#` | finds the prompt for you |
| `terminal length 0` (no `--More--`) | sends it for you |
| `show running-config` | `send_command("show running-config")` |
| copy the output from the screen | returns it as a Python string |
| `exit` | leaving the `with` block |

**Netmiko** = a Python library built on top of **Paramiko** (raw SSH) that knows how 100+ network operating systems behave
(Arista, Cisco, Juniper, ...). You pick the OS with `device_type`.

### Key words
| Word | Meaning |
|------|---------|
| **SSH** | Secure Shell - encrypted CLI login, TCP port 22 |
| **`.env` + dotenv** | Passwords live in `.env`, not in the code. `load_dotenv()` loads it, `os.getenv("NAME")` reads one value |
| **`os.getenv(name, default)`** | Returns the value, or the default if missing -> `os.getenv("SW1_PORT", 22)` |
| **device dict** | Everything Netmiko needs to log in: `device_type`, `host`, `username`, `password`, `port` |
| **`device_type`** | Which OS -> `"arista_eos"` (`cisco_ios`, `juniper_junos`, ...) |
| **`ConnectHandler`** | Opens the SSH session, logs in, prepares the CLI |
| **`with ... as conn`** | Context manager: closes the SSH session for you, even if something fails |
| **`send_command()`** | Runs ONE show command, waits for the prompt, returns the output as text |

---
## Setup (one time - in the terminal, not in the `.py` file)

Skip this if you already did it for the earlier scripts.

```bash
pip install -r requirements.txt
cp .env.example .env
```
Your `.env` must contain:
```
GEMINI_API_KEY=...   # not needed for this script
DEVICE_USERNAME=admin
DEVICE_PASSWORD=admin
SW1_HOST=172.20.20.11
SW2_HOST=172.20.20.12
# optional, default 22:
# SW1_PORT=22
# SW2_PORT=22
```
> `.env` holds secrets -> **never** share it, **never** commit it to git.

---
## Section 1 : Imports + load the `.env` file

- `os` -> read values from the environment with `os.getenv()`
- `load_dotenv()` -> copies every `KEY=value` line of `.env` into the environment
- `ConnectHandler` -> the Netmiko class that opens the SSH session

**Run it:** nothing is printed - no error = Netmiko is installed.

In [ ]:
# ---------------------------------------------------------------
# SECTION 1 : Imports + load the .env file
# ---------------------------------------------------------------
import os

from dotenv import load_dotenv
from netmiko import ConnectHandler

load_dotenv()

---
## Section 2 : One dictionary per switch

- All values come from `.env` -> the **same script works in every lab**, only `.env` changes.
- `int(...)` -> values from `.env` are always **text**; the port must be a **number**.
- `SWITCHES` is a dict of dicts: the name (`"sw1"`) -> everything needed to log in.

**Run it:** prints where the script will connect. Check the IPs and ports are yours.
(We never print the password.)

In [ ]:
# ---------------------------------------------------------------
# SECTION 2 : One dictionary per switch (values from .env)
# ---------------------------------------------------------------

USERNAME = os.getenv("DEVICE_USERNAME")
PASSWORD = os.getenv("DEVICE_PASSWORD")

SWITCHES = {
    "sw1": {
        "device_type": "arista_eos",
        "host": os.getenv("SW1_HOST"),
        "port": int(os.getenv("SW1_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
    "sw2": {
        "device_type": "arista_eos",
        "host": os.getenv("SW2_HOST"),
        "port": int(os.getenv("SW2_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
}

for name, device in SWITCHES.items():
    print(f"{name} -> {device['host']}:{device['port']} as {device['username']}")

---
## Section 3 : Quick test - SSH to sw1 and print its config

The shortest possible version, to prove SSH works:

- `ConnectHandler(**SWITCHES["sw1"])` -> `**` **unpacks** the dict into
  `ConnectHandler(device_type="arista_eos", host="...", port=..., ...)`
- `conn.enable()` -> go to privileged mode (`sw1#`). Needed on many devices before `show running-config`; harmless if already there.
- When the `with` block ends, the SSH session is closed automatically.

**Run it:** you should see the full running config of sw1.

> Once it works you can comment this section out (select + `Ctrl+/`) - Section 5 does the same for both switches.

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 : Quick test - SSH to sw1, print the running config
# ---------------------------------------------------------------
print("=" * 60)
print("QUICK TEST : sw1")
print("=" * 60)

with ConnectHandler(**SWITCHES["sw1"]) as conn:
    conn.enable()
    print(conn.send_command("show running-config"))

---
## Section 4 : A reusable function

Same code as Section 3, wrapped in a **function** -> write once, use for **both** switches
(and we reuse it unchanged in scripts 03 and 04).

- `def get_running_config(device):` -> takes ONE device dict
- `return` -> hands the config text back to the caller instead of printing it

**Run it:** nothing new is printed - a function only runs when you **call** it (next section).

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 : Function - SSH to ONE switch, return its running config
# ---------------------------------------------------------------

def get_running_config(device):
    with ConnectHandler(**device) as conn:
        conn.enable()
        return conn.send_command("show running-config")

---
## Section 5 : Loop over both switches + friendly errors

- `.items()` gives `(name, device)` pairs -> `("sw1", {...})`, `("sw2", {...})`
- `try / except` -> **one broken switch does not stop the whole script**
- `continue` -> skip to the next switch
- `len(config.splitlines())` -> number of lines in the config

**Run it:** both configs are printed, each with a line count.

**Try it yourself:** put a wrong IP for `SW2_HOST` in `.env` and run again - sw1 still works,
sw2 shows a one-line error. (Then fix it back.)

### Common errors
| Error | Usual cause |
|-------|-------------|
| `NetmikoTimeoutException` | Wrong IP / port, switch down, firewall. Try `ping` / `ssh` by hand |
| `NetmikoAuthenticationException` | Wrong username or password in `.env` |
| `host = None` / `ValueError` | `SW1_HOST` missing in `.env` |

In [ ]:
# ---------------------------------------------------------------
# SECTION 5 : Loop over both switches (one failure does not stop the rest)
# ---------------------------------------------------------------
for name, device in SWITCHES.items():
    print("=" * 60)
    print(f"{name}  ({device['host']}:{device['port']})")
    print("=" * 60)

    try:
        config = get_running_config(device)
    except Exception as error:
        print(f"Could not get the config -> {type(error).__name__}: {error}\n")
        continue

    print(config)
    print(f"\n[{name}: {len(config.splitlines())} lines, {len(config)} characters]\n")

---
## Recap - what you can now explain

- **Netmiko** turns a manual SSH session into 3 lines of Python: `ConnectHandler` -> `send_command` -> close.
- Device details and passwords live in **`.env`**, never in the code.
- A **device dict** + `**` unpacking is how Netmiko receives the login details.
- A **function** + a **loop** = the same task on 2 switches or 200.
- `try / except` keeps the script running when one device fails.

**Next:** Script 03 - give these configs to Gemini and ask questions about **your** network.